# Database Demonstration: Read, Insert, Update, and Delete in SQLite

**IDC409 Course Project — Topic 4: Web Scraping II (Faculty Profiles)**

This notebook satisfies **Course Requirement 2: Database — read, write, and modify**.

- **Table Verification:** Confirms all 4 SQLite tables generated by the project pipeline (`faculty_raw`, `faculty_clean`, `faculty_clusters`, `cluster_terms`).
- **Safe CRUD Demonstration:** Operations (`SELECT`, `INSERT`, `UPDATE`, `DELETE`) are demonstrated on an isolated copy (`demo_scratch.db`) which is cleaned up automatically, ensuring the production database remains untouched.

### Step 1: Verify Tables in the Production Database (`iisermohali_faculty.db`)

In [1]:
import os, shutil, sqlite3
import pandas as pd

real = sqlite3.connect("iisermohali_faculty.db")
print(pd.read_sql("SELECT name FROM sqlite_master WHERE type='table'", real))
real.close()

               name
0       faculty_raw
1     faculty_clean
2  faculty_clusters
3     cluster_terms


### Step 2: Safe CRUD Operations (Read, Write, Modify, Delete) on Scratch Database

In [2]:
shutil.copy("iisermohali_faculty.db", "demo_scratch.db")
con = sqlite3.connect("demo_scratch.db")
cur = con.cursor()

print("READ: faculty per department")
print(pd.read_sql("SELECT Department, COUNT(*) AS n FROM faculty_clean GROUP BY Department ORDER BY n DESC", con))

# WRITE: INSERT a new row
cur.execute("INSERT INTO faculty_clean (Name, Department, Designation, Category) VALUES (?, ?, ?, ?)",
            ("Test Person", "DMS", "Assistant Professor", "Regular Faculty"))
con.commit()
print("rows after INSERT:", cur.execute("SELECT COUNT(*) FROM faculty_clean").fetchone()[0])

# MODIFY: UPDATE that row
cur.execute("UPDATE faculty_clean SET Designation = ? WHERE Name = ?", ("Associate Professor", "Test Person"))
con.commit()
print(pd.read_sql("SELECT Name, Department, Designation FROM faculty_clean WHERE Name = 'Test Person'", con))

# DELETE it again
cur.execute("DELETE FROM faculty_clean WHERE Name = ?", ("Test Person",))
con.commit()
print("rows after DELETE:", cur.execute("SELECT COUNT(*) FROM faculty_clean").fetchone()[0])
con.close()
os.remove("demo_scratch.db")

real = sqlite3.connect("iisermohali_faculty.db")
print("real database still has", real.execute("SELECT COUNT(*) FROM faculty_clean").fetchone()[0], "rows")
real.close()

READ: faculty per department
  Department   n
0        DPS  32
1        DBS  31
2        DCS  26
3        DMS  23
4        EES  13
5        HSS  10
rows after INSERT: 136
          Name Department          Designation
0  Test Person        DMS  Associate Professor
rows after DELETE: 135
real database still has 135 rows
